In [ ]:
import json
import re

def extract_se_loger_data_from_file(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # 1. Extract JSON.parse(" ... ")
    match = re.search(r'JSON\.parse\("([\s\S]*?)"\)', text)
    if not match:
        return None

    raw = match.group(1)

    try:
        # 2. Unescape JS string safely using json decoding trick
        raw = json.loads(f'"{raw}"')

        # 3. Now parse real JSON
        data = json.loads(raw)

        location = data.get("app_cldp", {}) \
                      .get("data", {}) \
                      .get("classified", {}) \
                      .get("sections", {}) \
                      .get("location", {})

        if not location:
            return None

        return {
            "isAddressPublished": location.get("isAddressPublished"),
            "coordinates": location.get("geometry", {}).get("coordinates")
        }

    except json.JSONDecodeError as e:
        print("[ERROR] JSON decode failed:", e)
        return None